# 03 · Especialização: agente com ferramentas

**Critério da rubrica:** 05 (especialização, LoRA ou agente, comparada ao RAG simples).

Escolhemos o **agente** em vez do LoRA. O modelo é o mesmo do RAG; o que muda é que ele pode agir antes de responder:

| Ferramenta | Para quê |
|---|---|
| `buscar_base` | buscar de novo na base com outras palavras |
| `ler_resumo_loja` | ler o desperdício medido de uma loja no mês (formato do `GET /summary`) |
| `calcular_custo_desperdicio` | aplicar a fórmula de custo do produto |
| `calculadora` | fazer contas simples sem errar a aritmética |

O agente tem duas fases. Na primeira, parte da mesma busca do RAG simples e decide quais ferramentas chamar. Na segunda, escreve a resposta com o mesmo pedido fundamentado do RAG, agora com tudo o que reuniu. A citação passa pelo mesmo verificador.

In [1]:
import sys, json
from pathlib import Path
RAIZ = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(RAIZ / "src"))
import pandas as pd
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.width", 220)
from consultor import config
print("modelo:", config.MODELO_LLM, "| embeddings:", config.MODELO_EMBEDDING, "| limiar de recusa:", config.limiar_recusa())

modelo: qwen2.5:7b | embeddings: bge-m3 | limiar de recusa: 0.56


In [2]:
from consultor.agente import responder_agente
from consultor.geracao import responder_rag, MOTIVOS
from consultor import ferramentas
for f in ferramentas.ESPECIFICACAO:
    print(f["function"]["name"], "->", f["function"]["description"])

def mostrar(r):
    print(f"--- {r.config.upper()} | decisão: {r.motivo} | {r.segundos:.1f} s")
    for p in r.passos:
        origem = "busca inicial automática" if p.get("automatico") else "escolhida pelo modelo"
        resumo = json.dumps(p["resultado"], ensure_ascii=False)
        print(f"   ferramenta {p['ferramenta']}({json.dumps(p['argumentos'], ensure_ascii=False)}) [{origem}]")
        print(f"      -> {resumo[:230]}{'...' if len(resumo) > 230 else ''}")
    print(json.dumps(r.saida, ensure_ascii=False, indent=2)); print()

def comparar(pergunta):
    print("PERGUNTA:", pergunta, "\n")
    mostrar(responder_rag(pergunta)); mostrar(responder_agente(pergunta))

buscar_base -> Procura trechos na base de conhecimento sobre eficiência energética, refrigeração, iluminação, tarifas, bandeiras, conservação de alimentos e regras do produto.
ler_resumo_loja -> Lê o resumo de desperdício de uma loja em um mês: total em R$ e kWh e o detalhe por equipamento.
calcular_custo_desperdicio -> Aplica a fórmula do produto: custo = potencia_kw x fator_perda x horas x tarifa. Devolve kWh desperdiçados e custo em reais. Parâmetros omitidos usam o padrão do produto.
calculadora -> Calcula uma expressão aritmética com + - * / e parênteses. Exemplo: 1200 * 0.07877


## 1. Conta com a fórmula do produto

Resposta certa: 0,50 kW × 0,30 × 5 h × R$ 0,90 = **R$ 0,68** (0,75 kWh).

In [3]:
comparar("Quanto custa deixar a porta do freezer aberta por 5 horas no mês, com os valores padrão do produto?")

PERGUNTA: Quanto custa deixar a porta do freezer aberta por 5 horas no mês, com os valores padrão do produto? 



--- RAG | decisão: recusa_modelo | 3.3 s
{
  "nao_consta": true,
  "recomendacao": "",
  "fonte": null
}



--- AGENTE | decisão: resposta | 10.0 s
   ferramenta buscar_base({"consulta": "Quanto custa deixar a porta do freezer aberta por 5 horas no mês, com os valores padrão do produto?"}) [busca inicial automática]
      -> {"resultados": [{"trecho_id": "produto-contrato-03", "titulo": "Energia que dá Lucro — regra de alerta de porta aberta", "texto": "O sistema gera alerta quando a porta de um expositor ou freezer fica aberta por mais de 60 segundos...
   ferramenta calcular_custo_desperdicio({"duracao_valor": 5, "unidade": "h", "fator_perda": 0.3, "potencia_kw": 0.5, "tarifa_rs_kwh": 0.9}) [escolhida pelo modelo]
      -> {"duracao_s": 18000.0, "kwh_desperdicados": 0.75, "custo_rs": 0.675, "parametros": {"potencia_kw": 0.5, "fator_perda": 0.3, "tarifa_rs_kwh": 0.9}, "fonte": {"trecho_id": "produto-contrato-01", "titulo": "Fórmula do custo estimado"...
{
  "nao_consta": false,
  "recomendacao": "Deixar a porta do freezer aberta por 5 horas no mês custa R$ 0,675, com os valores padrão do pr

## 2. Conta com um valor que está na base

Resposta certa: 2.000 kWh × R$ 0,04463 = **R$ 89,26**.

In [4]:
comparar("Minha loja consome 2.000 kWh por mês. Quanto a bandeira vermelha patamar 1 acrescenta na conta?")

PERGUNTA: Minha loja consome 2.000 kWh por mês. Quanto a bandeira vermelha patamar 1 acrescenta na conta? 



--- RAG | decisão: resposta | 4.0 s
{
  "nao_consta": false,
  "recomendacao": "Com a bandeira vermelha patamar 1, o acréscimo na conta será de R$ 89,26.",
  "fonte": {
    "titulo": "Bandeiras Tarifárias",
    "trecho_id": "aneel-bandeiras-02"
  }
}



--- AGENTE | decisão: resposta | 9.1 s
   ferramenta buscar_base({"consulta": "Minha loja consome 2.000 kWh por mês. Quanto a bandeira vermelha patamar 1 acrescenta na conta?"}) [busca inicial automática]
      -> {"resultados": [{"trecho_id": "aneel-bandeiras-02", "titulo": "Bandeiras Tarifárias — o que significa cada cor e quanto custa", "texto": "Bandeira verde: condições favoráveis de geração de energia. A tarifa não sofre nenhum acrésc...
   ferramenta calculadora({"expressao": "2000 * 0.04463"}) [escolhida pelo modelo]
      -> {"expressao": "2000 * 0.04463", "resultado": 89.26}
{
  "nao_consta": false,
  "recomendacao": "Com base na bandeira vermelha patamar 1, o acréscimo na conta de energia será de R$ 89,26.",
  "fonte": {
    "titulo": "Bandeiras Tarifárias",
    "trecho_id": "aneel-bandeiras-02"
  }
}



## 3. Pergunta sobre uma loja

O RAG simples só enxerga a base, que não tem os números da loja. O agente lê o resumo. Resposta certa: a iluminação do estoque (`iluminacao-estoque`), R$ 3,23 e 3,59 kWh em novembro, mês ainda em andamento.

In [5]:
def confere(r, esperado):
    texto = r.saida["recomendacao"].lower()
    print(f">>> a resposta do {r.config} cita '{esperado}'?", esperado in texto, "\n")

pergunta = "Qual foi o maior desperdício do mercado-01 em 2026-11 e quanto custou?"
comparar(pergunta)
confere(responder_agente(pergunta), "estoque")

PERGUNTA: Qual foi o maior desperdício do mercado-01 em 2026-11 e quanto custou? 



--- RAG | decisão: recusa_limiar | 0.0 s
{
  "nao_consta": true,
  "recomendacao": "",
  "fonte": null
}



--- AGENTE | decisão: resposta | 14.0 s
   ferramenta buscar_base({"consulta": "Qual foi o maior desperdício do mercado-01 em 2026-11 e quanto custou?"}) [busca inicial automática]
      -> {"resultados": [], "aviso": "Nenhum trecho relevante na base para esta consulta."}
   ferramenta ler_resumo_loja({"loja_id": "mercado-01", "periodo": "2026-11"}) [escolhida pelo modelo]
      -> {"resumo": {"loja_id": "mercado-01", "periodo": "2026-11", "total_eventos": 276, "total_kwh": 6.56, "total_custo_rs": 5.91, "por_equipamento": [{"equipamento_id": "iluminacao-estoque", "tipo": "luz_acesa_sem_necessidade", "eventos...
   ferramenta calcular_custo_desperdicio({"duracao_valor": 20164, "unidade": "s"}) [escolhida pelo modelo]
      -> {"duracao_s": 20164.0, "kwh_desperdicados": 0.8402, "custo_rs": 0.7561, "parametros": {"potencia_kw": 0.5, "fator_perda": 0.3, "tarifa_rs_kwh": 0.9}, "fonte": {"trecho_id": "produto-contrato-01", "titulo": "Fórmula do custo estima...
{
  "nao_consta": false,
  "rec

>>> a resposta do agente cita 'estoque'? True 



### Um caso em que o agente erra

A mesma pergunta, trocando "maior desperdício" por "equipamento que mais desperdiçou". A resposta certa continua sendo a iluminação do estoque. A célula mostra o que o agente faz e confere o resultado. Este caso entrou na lista de problemas observados do relatório.

In [6]:
pergunta = "Qual equipamento do mercado-01 mais desperdiçou em 2026-11 e quanto custou?"
r = responder_agente(pergunta)
print("PERGUNTA:", pergunta, "\n"); mostrar(r)
confere(r, "estoque")

PERGUNTA: Qual equipamento do mercado-01 mais desperdiçou em 2026-11 e quanto custou? 

--- AGENTE | decisão: resposta | 10.0 s
   ferramenta buscar_base({"consulta": "Qual equipamento do mercado-01 mais desperdiçou em 2026-11 e quanto custou?"}) [busca inicial automática]
      -> {"resultados": [], "aviso": "Nenhum trecho relevante na base para esta consulta."}
   ferramenta ler_resumo_loja({"loja_id": "mercado-01", "periodo": "2026-11"}) [escolhida pelo modelo]
      -> {"resumo": {"loja_id": "mercado-01", "periodo": "2026-11", "total_eventos": 276, "total_kwh": 6.56, "total_custo_rs": 5.91, "por_equipamento": [{"equipamento_id": "iluminacao-estoque", "tipo": "luz_acesa_sem_necessidade", "eventos...
{
  "nao_consta": false,
  "recomendacao": "O freezer-2 foi o equipamento que mais desperdiçou energia, com 0.84 kWh e custo de 0.76 R$ em 2026-11. Sugerimos verificar se há necessidade de manter a porta aberta tão frequentemente.",
  "fonte": {
    "titulo": "Resumo de desperdício da lo

## 4. O agente também precisa saber recusar

In [7]:
comparar("Quanto custa contratar um eletricista para trocar a fiação da loja?")

PERGUNTA: Quanto custa contratar um eletricista para trocar a fiação da loja? 



--- RAG | decisão: recusa_limiar | 0.0 s
{
  "nao_consta": true,
  "recomendacao": "",
  "fonte": null
}



--- AGENTE | decisão: recusa_sem_evidencia | 5.3 s
   ferramenta buscar_base({"consulta": "Quanto custa contratar um eletricista para trocar a fiação da loja?"}) [busca inicial automática]
      -> {"resultados": [], "aviso": "Nenhum trecho relevante na base para esta consulta."}
   ferramenta buscar_base({"consulta": "custo de eletricista para trocar fiação"}) [escolhida pelo modelo]
      -> {"resultados": [], "aviso": "Nenhum trecho relevante na base para esta consulta."}
{
  "nao_consta": true,
  "recomendacao": "",
  "fonte": null
}



## 5. Comparação no conjunto de teste

Os números abaixo vêm de `avaliacao/resultados/`, gravados por `python scripts/avaliar.py`. O conjunto de teste tem 44 casos que não foram usados para ajustar prompts nem limiar.

In [8]:
from consultor import avaliacao
def carregar(config_nome):
    return [json.loads(l) for l in open(RAIZ / "avaliacao" / "resultados" / f"teste_{config_nome}.jsonl", encoding="utf-8")]
linhas = []
for nome in ("rag", "agente"):
    correcoes = [r["correcao"] for r in carregar(nome)]
    resumo = avaliacao.resumir(correcoes)
    linhas.append({"configuração": nome, "acerto %": resumo["acerto_%"], "alucinação %": resumo["alucinacao_%"],
                   "factual %": resumo["acerto_factual_%"], "cálculo %": resumo["acerto_calculo_%"], "loja %": resumo["acerto_loja_%"],
                   "não consta %": resumo["acerto_nao_consta_%"], "recusa indevida %": resumo["recusa_indevida_%"], "tempo médio (s)": resumo["segundos_medio"]})
display(pd.DataFrame(linhas))

,configuração,acerto %,alucinação %,factual %,cálculo %,loja %,não consta %,recusa indevida %,tempo médio (s)
0,rag,72.7,2.3,90.0,33.3,0.0,100.0,34.4,2.21
1,agente,90.9,4.5,95.0,66.7,83.3,100.0,6.2,8.41


In [9]:
rag = {r["caso"]: r for r in carregar("rag")}; agente = {r["caso"]: r for r in carregar("agente")}
ganhos = [c for c in rag if agente[c]["correcao"]["acerto"] and not rag[c]["correcao"]["acerto"]]
perdas = [c for c in rag if rag[c]["correcao"]["acerto"] and not agente[c]["correcao"]["acerto"]]
print(f"casos que só o agente acertou ({len(ganhos)}):")
for c in ganhos:
    print(f"  {c} [{rag[c]['correcao']['categoria']}] {rag[c]['pergunta']}")
print(f"casos que só o RAG simples acertou ({len(perdas)}):")
for c in perdas:
    print(f"  {c} [{rag[c]['correcao']['categoria']}] {rag[c]['pergunta']}")
usos = pd.Series([p["ferramenta"] for r in agente.values() for p in r["resposta"]["passos"] if not p.get("automatico")]).value_counts()
print("\nferramentas que o agente escolheu chamar no teste:"); print(usos.to_string())

casos que só o agente acertou (8):
  t16 [factual] Como evitar que a luz do estoque fique acesa sem ninguém lá dentro?
  t24 [calculo] Somando o mês todo, os expositores ficaram 40 horas com a porta aberta. Qual é o desperdício estimado em reais com os valores padrão do produto?
  t25 [calculo] Qual é o custo estimado de 10 horas de porta aberta em um freezer se a tarifa for R$ 1,10 por kWh e os demais valores forem os padrão do produto?
  t27 [loja] Qual equipamento da padaria-01 mais desperdiçou energia em 2026-10 e o que devo fazer?
  t28 [loja] Qual foi o maior desperdício do mercado-01 em 2026-10 e qual é a recomendação?
  t29 [loja] Quanto a padaria-01 desperdiçou em reais em 2026-10?
  t30 [loja] Quantos kWh o mercado-01 desperdiçou em 2026-10?
  t31 [loja] Na padaria-01, em 2026-10, quem desperdiçou mais: o freezer-1 ou o expositor-1?
casos que só o RAG simples acertou (0):

ferramentas que o agente escolheu chamar no teste:
buscar_base                   26
ler_resumo_loja     